# Analysis-period ensemble map metrics

Maps of ensemble verification metrics (bias, RMSE, spread, CRPS, ...) for the atmospheric analysis (`da`) runs against their references, one figure per case, variable and metric. Global scalar metrics and rank histograms are in `01_compute_and_plot_scalar_metrics.ipynb`.

Cases: **S0** (Dec 2011) compares CTRL and EAM-DART with 10/20/40 members; **S1** (May 2012) has EAM-DART with 40 members. Model fields are monthly means of the regridded 6-hourly series.

**Period:** these monthly-mean diagnostics use the whole calendar month (S0: Dec 2011; S1: May 2012), including the first ~15 days of DA spin-up, because their references (ERA5, GPCP, CERES-OAFlux) are monthly products. The analysis-period scorecard (`04_compute_da_scorecard_metrics.ipynb` / `05_plot_da_scorecards.ipynb`) instead uses the 64 assimilation cycles of 16-31 Dec / 16-31 May with 6-hourly and daily references; use it for the with/without-DA comparison.

## 1. Settings

In [ ]:
from pathlib import Path

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
workflow_name = "analysis_atm_model_evaluation"

component = "atm"
run_segment = "da"
# Analysis cases: season -> experiments compared together (same period).
cases = {
    "S0": ("CTRL10-S0", "DART10-S0", "DART20-S0", "DART40-S0"),  # 201112
    "S1": ("DART40-S1",),  # 201205
}
variables = ("TS", "TREFHT", "PSL", "TMQ", "PRECT", "FLUT")
map_metrics = ("bias", "rmse", "spread", "crps")
n_bootstrap = 0  # > 0 adds 95% ensemble-bootstrap CIs (hatching where bias is significant)

# Model input averaged to monthly means. None -> per-variable default for
# atmosphere analyses: state variables (TS, TREFHT, PSL, TMQ) use the analysis
# values at 00/06/12/18 UTC ("6hourly_da"); flux/process variables (PRECT,
# FLUT) use the 6-h forecast means after each analysis ("6hourly"). A string
# here ("6hourly", "6hourly_da", "clim") applies to every variable.
model_source = None
lat_range = (-90, 90)
lon_range = (-180, 180)
max_members = None  # e.g. 2 for a quick test; caches record the ensemble size
random_seed = 42
# Caches are rebuilt automatically when they are missing, older than any
# model/reference input, or were built with other settings; force_compute
# rebuilds them regardless.
force_compute = False
show = True
FIGURE_DPI = 200

## 2. Project setup

In [ ]:
import sys
import matplotlib.pyplot as plt

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.model_evaluation import MAP_METRICS, evaluate_cases, plot_evaluation_maps

data_dir, figure_dir = workflow_output_dirs(
    workflow_name, output_root=DIAGNOSTIC_OUTPUT_ROOT
)
unknown = set(map_metrics) - set(MAP_METRICS)
if unknown:
    raise ValueError(f"Unknown map metrics {sorted(unknown)}; choose from {MAP_METRICS}")

## 3. Compute or reuse map metrics

In [ ]:
results, registries, failures = evaluate_cases(
    "map", cases, variables, data_root=INPUT_DATA_ROOT, cache_dir=data_dir,
    component=component, run_segment=run_segment, model_source=model_source,
    n_bootstrap=n_bootstrap, random_seed=random_seed,
    max_members=max_members, lat_range=lat_range, lon_range=lon_range,
    force_compute=force_compute,
)
if failures:
    print(f"\n{len(failures)} case(s) failed and are skipped below:")
    for season, variable, experiment, error in failures:
        print(f"  {season} {variable} {experiment}: {error}")

## 4. Plot, save, and show

In [ ]:
for season, by_variable in results.items():
    registry = registries[season]
    for variable, datasets in by_variable.items():
        period = registry[next(iter(datasets))]["period"]
        for metric in map_metrics:
            fig = plot_evaluation_maps(
                datasets, metric, variable,
                label_map={name: registry[name]["label"] for name in datasets},
                title=f"{variable}: {metric} ({season} {run_segment}, {period})",
            )
            output = figure_dir / f"{variable}_{metric}_{season}_{run_segment}.png"
            fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
            print(f"[FIGURE] {output}")
            if not show:
                plt.close(fig)

## Notes: variables and map metrics

Atmosphere variables: `FLUT` (CERES-OAFlux), `PRECT` (GPCP), `TMQ`, `TS`, `TREFHT`, and `PSL` (ERA5).

| Metric | Definition |
|---|---|
| `mean`, `reference_mean` | time-mean ensemble mean / reference |
| `bias` | ensemble mean minus reference, time mean |
| `bias_stddev` | temporal standard deviation of the ensemble-mean error |
| `rmse` | RMSE of individual members over members and time |
| `spread` | ensemble standard deviation, time mean |
| `crps` | continuous ranked probability score, time mean |
| `coverage` | fraction of times the reference lies inside the ensemble envelope |

Caches: `<experiment>_<variable>_<period>_map_metrics.nc` in the workflow data directory.